In [18]:
import os 
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import START , END, StateGraph
from typing import TypedDict, Annotated
from pydantic import BaseModel , Field
import time
from operator import add
from functools import reduce
load_dotenv()
model = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.7)

In [19]:
class EvaluationSchema(BaseModel) :
    feedback  :str = Field(description="Provide feedback for essay")
    score     :int = Field(description="Score out of 10", ge=0, le=10)

structured_model =  model.with_structured_output(EvaluationSchema)

In [20]:
input_essay = """Pakistan's role in Artificial Intelligence (AI) is evolving, with both opportunities and challenges ahead. The country is actively working to integrate AI across various sectors, including education, governance, and national security, with the aim of driving economic growth and societal development. However, challenges like infrastructure deficits, a shortage of skilled professionals, and the need for robust ethical frameworks need to be addressed to fully realize AI's potential. 
AI's Role in Pakistan:
Economic Transformation:
Pakistan is aiming to leverage AI to transform its economy by boosting various sectors, including agriculture, healthcare, and manufacturing. 
Education:
AI is being explored to personalize learning experiences, bridge the urban-rural divide, and empower teachers. Initiatives like DigiSkills and collaborations with platforms like Khan Academy are examples of this integration. 
National Security:
AI is seen as crucial for enhancing national security by managing data, analyzing patterns, and predicting risks, particularly in war-prone areas. 
Public Service Delivery:
AI can potentially improve public service delivery by optimizing resource allocation, streamlining processes, and enhancing citizen engagement. 
Governance:
AI can be integrated into governance structures to improve decision-making, enhance transparency, and promote efficiency. 
Challenges:
Shortage of Skilled Professionals:
Pakistan faces a shortage of skilled professionals in AI, which is hindering the country's ability to fully utilize AI technologies. 
Infrastructure Deficits:
Limited access to reliable internet and computing infrastructure poses a barrier to AI adoption across the country. 
Ethical Concerns:
As AI systems become more sophisticated, Pakistan needs to address ethical concerns related to data privacy, bias, and the potential for misuse. 
Policy Gaps:
A lack of comprehensive AI policies and regulations can slow down the pace of AI adoption and create uncertainty for researchers and businesses. 
Way Forward:
Invest in Education and Training:
Pakistan needs to invest in education and training programs to equip its workforce with the necessary AI skills. 
Develop Infrastructure:
Investing in robust digital infrastructure is crucial for supporting AI development and deployment. 
Establish Ethical Frameworks:
Pakistan needs to develop clear ethical guidelines and regulations for AI to ensure responsible development and deployment. 
Foster International Collaboration:
Strengthening international collaborations with leading AI research institutions can help Pakistan stay abreast of the latest advancements and best practices. 
By addressing these challenges and capitalizing on the opportunities, Pakistan can harness the transformative potential of AI to drive economic growth, improve public services, and enhance national security. """

In [21]:
prompt = f"Evaluate language quality of the essay and provide the feedback and assign a score out of 10.\n {essay}"
resp = structured_model.invoke(prompt)
print(resp)

feedback="The essay provides a good overview of Pakistan's evolving role in AI, covering both opportunities and challenges. It is well-structured and touches upon key areas such as economic transformation, education, national security, and governance. The challenges and way forward are also clearly outlined. However, the essay could benefit from more specific examples and in-depth analysis to strengthen its arguments. Additionally, refining the language to enhance clarity and conciseness would improve its overall impact." score=7


In [30]:
class Essay_evaluator(TypedDict):
    essay_text : str
    cot_feedback : str
    doa_feedback : str
    language_feedback  :str
    scores : Annotated[list[int], add]
    final_feedback : str
    avg_score : float

In [29]:
def cot(state : Essay_evaluator):
    essay = state["essay_text"]
    prompt = f"Assess the clarity of thought in the following essay and provide the feedback and assign a score out of 10.\n {essay}"
    result = structured_model.invoke(prompt)
    feedback, score = result.feedback , [result.score]
    return {"cot_feedback" : feedback, 'scores' : score}
def language(state : Essay_evaluator):
    essay = state["essay_text"]
    prompt = f"Evaluate language quality of the essay and provide the feedback and assign a score out of 10.\n {essay}"
    result = structured_model.invoke(prompt)
    feedback, score = result.feedback , [result.score]
    return {"language_feedback" : feedback, 'scores' : score}
    
def doa(state : Essay_evaluator):
    essay = state["essay_text"]
    prompt = f"Evaluate the depth of analysis in the following essay and provide the feedback and assign a score out of 10.\n {essay}"
    result = structured_model.invoke(prompt)
    feedback, score = result.feedback , [result.score]
    return {"doa_feedback" : feedback, 'scores' : score}
    
def final_evaluation(state : Essay_evaluator) -> Essay_evaluator:
    language_feedback  =state["language_feedback"]
    clarity_feedback = state["cot_feedback"]
    analysis_feedback = state["doa_feedback"]
    summary_prompt = f"""
Given the following feedback:
- Language: {language_feedback}
- Clarity of Thought: {clarity_feedback}
- Depth of Analysis: {analysis_feedback}

Write a concise summary that synthesizes these into an overall evaluation of the essay.
"""
    response = model.invoke(summary_prompt)
    avg = reduce(add, state["scores"]) / len(state["scores"])
    return {"final_feedback" : response , "avg_score" : avg}

In [31]:
graph = StateGraph(Essay_evaluator)
# Nodes
graph.add_node("cot", cot)
graph.add_node("doa", doa)
graph.add_node("language", language)
graph.add_node("final_evaluation", final_evaluation)

# Edges
graph.add_edge(START, "cot")
graph.add_edge(START, "doa")
graph.add_edge(START, "language")
graph.add_edge( "cot", "final_evaluation")
graph.add_edge( "doa", "final_evaluation")
graph.add_edge( "language", "final_evaluation")
graph.add_edge("final_evaluation", END)

workflow = graph.compile()

In [32]:
initial_state = {
    "essay_text" : input_essay
}
final_state = workflow.invoke(initial_state)
print(final_state)


{'essay_text': "Pakistan's role in Artificial Intelligence (AI) is evolving, with both opportunities and challenges ahead. The country is actively working to integrate AI across various sectors, including education, governance, and national security, with the aim of driving economic growth and societal development. However, challenges like infrastructure deficits, a shortage of skilled professionals, and the need for robust ethical frameworks need to be addressed to fully realize AI's potential. \nAI's Role in Pakistan:\nEconomic Transformation:\nPakistan is aiming to leverage AI to transform its economy by boosting various sectors, including agriculture, healthcare, and manufacturing. \nEducation:\nAI is being explored to personalize learning experiences, bridge the urban-rural divide, and empower teachers. Initiatives like DigiSkills and collaborations with platforms like Khan Academy are examples of this integration. \nNational Security:\nAI is seen as crucial for enhancing national